<a href="https://colab.research.google.com/github/Henrixfs/Proyecto_ML_Seguridad/blob/main/DATA_PROCESSING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Montar Google Drive para guardar los resultados allí
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils import resample
import torch
import joblib
import os

Mounted at /content/drive


In [ ]:
# Crear carpeta en Drive para guardar todo
output_dir = '/content/drive/MyDrive/Proyecto_ML_Seguridad'
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Carpeta creada/verificada en Drive: {output_dir}")

📁 Carpeta creada/verificada en Drive: /content/drive/MyDrive/Proyecto_ML_Seguridad


In [ ]:
print("1. Cargando y limpiando datasets...")
# Cargar SQL Dataset
df_sql = pd.read_csv('/content/drive/MyDrive/DATASET/Modified_SQL_Dataset.csv/Modified_SQL_Dataset.csv')[['Query', 'Label']].rename(columns={'Query': 'text', 'Label': 'label'})
df_sql = df_sql.dropna()

# Cargar XSS Dataset
df_xss = pd.read_csv('/content/drive/MyDrive/DATASET/XSS_dataset.csv/XSS_dataset.csv')[['Sentence', 'Label']].rename(columns={'Sentence': 'text', 'Label': 'label'})
df_xss = df_xss.dropna()

1. Cargando y limpiando datasets...


In [ ]:
print("2. Unificando y asignando clases (Normal=0, SQLi=1, XSS=2)...")
# Separar por tipo de ataque y tráfico normal
sql_normal = df_sql[df_sql['label'] == 0].copy()
sql_attack = df_sql[df_sql['label'] == 1].copy()
sql_attack['label'] = 1  # Confirmar clase 1 (SQLi)

xss_normal = df_xss[df_xss['label'] == 0].copy()
xss_attack = df_xss[df_xss['label'] == 1].copy()
xss_attack['label'] = 2  # Mapear XSS a la clase 2

# Juntar todo el tráfico normal (0)
df_normal_all = pd.concat([sql_normal, xss_normal], ignore_index=True)
df_normal_all['label'] = 0

2. Unificando y asignando clases (Normal=0, SQLi=1, XSS=2)...


In [ ]:
print("3. Balanceando las clases...")
# Buscar la cantidad mínima para igualar todas las clases
min_len = min(len(df_normal_all), len(sql_attack), len(xss_attack))
print(f"   -> Ajustando a {min_len} muestras por clase.")

df_normal_bal = resample(df_normal_all, replace=False, n_samples=min_len, random_state=42)
df_sqli_bal = resample(sql_attack, replace=False, n_samples=min_len, random_state=42)
df_xss_bal = resample(xss_attack, replace=False, n_samples=min_len, random_state=42)

# Unificar todo ya balanceado y mezclar (shuffle)
df_balanced = pd.concat([df_normal_bal, df_sqli_bal, df_xss_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Guardar el CSV Final en tu Google Drive
dataset_path = f'{output_dir}/dataset_final.csv'
df_balanced.to_csv(dataset_path, index=False)
print(f"✅ dataset_final.csv guardado con éxito en {dataset_path}")

print("4. Dividiendo datos (Train=70%, Validation=15%, Test=15%)...")
X = df_balanced['text']
y = df_balanced['label']

# División estratificada para mantener la proporción de clases
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

3. Balanceando las clases...
   -> Ajustando a 7373 muestras por clase.
✅ dataset_final.csv guardado con éxito en /content/drive/MyDrive/Proyecto_ML_Seguridad/dataset_final.csv
4. Dividiendo datos (Train=70%, Validation=15%, Test=15%)...


In [ ]:
print("5. Vectorizando con TF-IDF (N-gramas de caracteres) y generando Tensores...")
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3), max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train).toarray()
X_val_tfidf = vectorizer.transform(X_val).toarray()
X_test_tfidf = vectorizer.transform(X_test).toarray()

# Guardar Vectorizador para el Integrante 3 (Backend)
joblib.dump(vectorizer, f'{output_dir}/tfidf_vectorizer.pkl')

# Convertir a Tensores de PyTorch
X_train_tensor = torch.tensor(X_train_tfidf, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

X_val_tensor = torch.tensor(X_val_tfidf, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_tfidf, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# Guardar Tensores para el Integrante 2 (PyTorch) en Drive
torch.save((X_train_tensor, y_train_tensor), f'{output_dir}/train_data.pt')
torch.save((X_val_tensor, y_val_tensor), f'{output_dir}/val_data.pt')
torch.save((X_test_tensor, y_test_tensor), f'{output_dir}/test_data.pt')

print(f"✅ ¡Proceso completado! Todos los archivos (CSV, Vectorizador y Tensores .pt) están listos en tu carpeta de Drive: {output_dir}")

5. Vectorizando con TF-IDF (N-gramas de caracteres) y generando Tensores...
✅ ¡Proceso completado! Todos los archivos (CSV, Vectorizador y Tensores .pt) están listos en tu carpeta de Drive: /content/drive/MyDrive/Proyecto_ML_Seguridad
